## Create a drilling campaign object

This example shows how to publish a drilling campaign geoscience object to Evo from the sample CSV files in `sample-data`.

Drilling campaigns do not currently have a high-level typed wrapper like `PointSet`, so this notebook uses the current recommended schema-based workflow with `ObjectAPIClient`, reusable helper functions, and the `evo.widgets` notebook extension.

### Requirements

You must have a Seequent account with the Evo entitlement and an Evo application with a client ID and redirect URL.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [ ]:
%load_ext evo.widgets

from evo.notebooks import ServiceManagerWidget

input_path = "sample-data"

# Evo app credentials
client_id = "<your-client-id>"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(
    redirect_url=redirect_url,
    client_id=client_id,
).login()

### 1. Load the Current Drilling Campaign SDK

Unlike pointsets, drilling campaigns are still assembled from the schema models rather than a high-level typed object wrapper. This notebook keeps the newer tutorial style by combining `ObjectAPIClient`, `evo.widgets`, and a smaller set of reusable helper functions.

In [ ]:
import uuid

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from evo.notebooks import FeedbackWidget
from evo.objects import ObjectAPIClient
from evo.objects.utils.table_formats import (
    FLOAT_ARRAY_3,
    FLOAT_ARRAY_6,
    INTEGER_ARRAY_1_INT32,
    LOOKUP_TABLE_INT32,
    STRING_ARRAY,
 )
from evo.widgets import get_viewer_url_for_objects
from evo_schemas.components import (
    BoundingBox_V1_0_1,
    CategoryAttribute_V1_1_0,
    CategoryData_V1_0_1,
    Crs_V1_0_1_EpsgCode,
    DownholeDirectionVector_V1_0_0,
    HoleChunks_V1_0_0,
    HoleCollars_V1_0_0,
    NanCategorical_V1_0_1,
    StringAttribute_V1_1_0,
 )
from evo_schemas.elements import FloatArray3_V1_0_1, IntegerArray1_V1_0_1, LookupTable_V1_0_1, StringArray_V1_0_1
from evo_schemas.objects import (
    DrillingCampaign_V1_0_0,
    DrillingCampaign_V1_0_0_Interim,
    DrillingCampaign_V1_0_0_Planned,
    DrillingCampaign_V1_0_0_Planned_Path_NaturalDeviation,
 )

object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())
data_client = object_client.get_data_client(manager.cache)

### 2. Load the Input Data

The sample data includes one table for the planned collar and natural deviation inputs, and one table for the interim survey path. We load both up front so the rest of the notebook can build the full drilling campaign in one pass.

In [ ]:
planned_df = pd.read_csv(f"{input_path}/planned.csv")
interim_df = pd.read_csv(f"{input_path}/interim.csv")

print(f"Loaded {len(planned_df)} planned collar rows across {planned_df['Holeid'].nunique()} holes.")
print(f"Loaded {len(interim_df)} interim survey rows across {interim_df['Holeid'].nunique()} holes.")

display(planned_df.head())
display(interim_df.head())

### 3. Define Object Metadata

Set the key metadata for the new drilling campaign. The same hole ID column must be used across both input tables so the planned and interim sections can reference the same holes.

- `object_hole_id`: The column that identifies each hole in both CSV files.
- `object_name`: The display name of the drilling campaign object.
- `object_path`: The Evo workspace path where the object will be created.
- `object_epsg_code`: The EPSG code for the object CRS. Leave it as `None` if you do not want to assign a CRS.
- `object_tags`: Optional tags to store alongside the object.

In [ ]:
# The same hole ID column must be used across the planned and interim input tables.
object_hole_id = "Holeid"

object_name = "Drilling campaign SDK demo"
object_path = "Jupyter_Example"
object_epsg_code = 32650
object_tags = {"Source": "Jupyter Notebook"}

full_obj_path = f"{object_path}/{object_name}.json"

coordinate_reference_system = (
    Crs_V1_0_1_EpsgCode(epsg_code=object_epsg_code) if object_epsg_code is not None else None
)

planned_attribute_specs = [
    {"name": "Comment", "key": str(uuid.uuid4()), "type": "string"},
]
interim_attribute_specs = [
    {"name": "Wolfpass GM", "key": str(uuid.uuid4()), "type": "category"},
]

### 4. Define Reusable Helpers

These helpers keep the notebook closer to the newer typed examples by centralizing the repeated schema assembly work in one place. The rest of the notebook can then focus on the data flow rather than the low-level table wiring.

In [ ]:
def save_float_array3(dataframe: pd.DataFrame) -> dict:
    return data_client.save_dataframe(dataframe.astype("float64"), table_format=FLOAT_ARRAY_3)


def save_float_array6(dataframe: pd.DataFrame) -> dict:
    return data_client.save_dataframe(dataframe.astype("float64"), table_format=FLOAT_ARRAY_6)


def create_hole_id_mapping(hole_id_table: pd.DataFrame, value_list: pd.DataFrame) -> pd.DataFrame:
    mapping_df = pd.DataFrame(
        {
            "hole_index": hole_id_table["key"].astype("int32"),
            "offset": pd.Series(0, index=hole_id_table.index, dtype="uint64"),
            "count": pd.Series(0, index=hole_id_table.index, dtype="uint64"),
        }
    )

    offset = 0
    for hole_id, group in value_list.groupby("data", sort=False):
        match = hole_id_table.loc[hole_id_table["value"] == hole_id, "key"]
        if match.empty:
            continue

        mapping_df.loc[mapping_df["hole_index"] == int(match.iloc[0]), ["offset", "count"]] = (
            np.uint64(offset),
            np.uint64(len(group)),
        )
        offset += len(group)

    return mapping_df


def create_category_lookup_and_values(series: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned = series.fillna("").astype("string")
    unique_values = sorted(cleaned.unique().tolist())
    table_df = pd.DataFrame(
        {
            "key": pd.Series(range(len(unique_values)), dtype="int32"),
            "value": unique_values,
        }
    )
    value_lookup = table_df.set_index("value")["key"]
    values_df = pd.DataFrame({"data": cleaned.map(value_lookup).astype("int32")})
    return table_df, values_df


def build_attribute_components(frame: pd.DataFrame, attribute_specs: list[dict]) -> list:
    attributes = []
    for attribute_spec in attribute_specs:
        name = attribute_spec["name"]
        key = attribute_spec["key"]
        attribute_type = attribute_spec["type"]

        if attribute_type == "string":
            values_df = pd.DataFrame({"data": frame[name].fillna("").astype("string")})
            values = StringArray_V1_0_1.from_dict(data_client.save_dataframe(values_df, table_format=STRING_ARRAY))
            attribute = StringAttribute_V1_1_0(name=name, key=key, values=values)
        elif attribute_type == "category":
            table_df, values_df = create_category_lookup_and_values(frame[name])
            table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df, table_format=LOOKUP_TABLE_INT32))
            values = IntegerArray1_V1_0_1.from_dict(
                data_client.save_dataframe(values_df, table_format=INTEGER_ARRAY_1_INT32)
            )
            attribute = CategoryAttribute_V1_1_0(
                name=name,
                key=key,
                nan_description=NanCategorical_V1_0_1(values=[]),
                table=table,
                values=values,
            )
        else:
            raise ValueError(f"Unsupported attribute type: {attribute_type}")

        attributes.append(attribute)

    return attributes


def create_hole_id_component(sorted_frame: pd.DataFrame, hole_id_column: str) -> tuple[pd.DataFrame, CategoryData_V1_0_1]:
    hole_id_table_df = pd.DataFrame(
        {
            "key": pd.Series(range(len(sorted_frame)), dtype="int32"),
            "value": sorted_frame[hole_id_column].astype("string"),
        }
    )
    hole_id_values_df = pd.DataFrame({"data": pd.Series(range(len(sorted_frame)), dtype="int32")})

    hole_id_table_component = LookupTable_V1_0_1.from_dict(
        data_client.save_dataframe(hole_id_table_df, table_format=LOOKUP_TABLE_INT32)
    )
    hole_id_values_component = IntegerArray1_V1_0_1.from_dict(
        data_client.save_dataframe(hole_id_values_df, table_format=INTEGER_ARRAY_1_INT32)
    )

    hole_id_component = CategoryData_V1_0_1(table=hole_id_table_component, values=hole_id_values_component)
    return hole_id_table_df, hole_id_component


def build_bounding_box(coordinates_df: pd.DataFrame) -> BoundingBox_V1_0_1:
    return BoundingBox_V1_0_1(
        min_x=coordinates_df["x"].min(),
        max_x=coordinates_df["x"].max(),
        min_y=coordinates_df["y"].min(),
        max_y=coordinates_df["y"].max(),
        min_z=coordinates_df["z"].min(),
        max_z=coordinates_df["z"].max(),
    )

### 5. Build the Planned Section

The planned section uses the collar table to define the shared hole IDs, collar locations, distances, collar attributes, and the natural deviation path. Those pieces are then combined into the planned component for the drilling campaign.

In [ ]:
sorted_planned_df = planned_df.sort_values(object_hole_id).reset_index(drop=True)

hole_id_table_df, hole_id_component = create_hole_id_component(sorted_planned_df, object_hole_id)

coordinates_df = sorted_planned_df[["Easting", "Northing", "Elevation"]].rename(
    columns={"Easting": "x", "Northing": "y", "Elevation": "z"}
 )
bounding_box = build_bounding_box(coordinates_df)
coordinates_component = FloatArray3_V1_0_1.from_dict(save_float_array3(coordinates_df))

distances_df = pd.DataFrame(
    {
        "final": pd.to_numeric(sorted_planned_df["Target Depth"], errors="coerce")
        + pd.to_numeric(sorted_planned_df["Extension"], errors="coerce"),
        "target": pd.to_numeric(sorted_planned_df["Target Depth"], errors="coerce"),
        "current": np.zeros(len(sorted_planned_df), dtype="float64"),
    }
 )
distances_component = FloatArray3_V1_0_1.from_dict(save_float_array3(distances_df))

planned_hole_values_df = pd.DataFrame({"data": sorted_planned_df[object_hole_id].astype("string")})
planned_hole_chunks_df = create_hole_id_mapping(hole_id_table_df, planned_hole_values_df)
planned_hole_chunks_component = HoleChunks_V1_0_0.from_dict(data_client.save_dataframe(planned_hole_chunks_df))

planned_collar_attributes = build_attribute_components(sorted_planned_df, planned_attribute_specs)
planned_collar_component = HoleCollars_V1_0_0(
    coordinates=coordinates_component,
    distances=distances_component,
    holes=planned_hole_chunks_component,
    attributes=planned_collar_attributes,
 )

natural_deviation_df = sorted_planned_df[["Extension", "Azimuth", "Dip", "Lift Rate", "Drift Rate", "Distance"]].rename(
    columns={
        "Extension": "distance",
        "Azimuth": "azimuth",
        "Dip": "dip",
        "Lift Rate": "lift rate",
        "Drift Rate": "drift rate",
        "Distance": "deviation rate distance",
    }
 )
natural_deviation_df["distance"] = pd.to_numeric(sorted_planned_df["Target Depth"], errors="coerce") + pd.to_numeric(
    sorted_planned_df["Extension"], errors="coerce"
 )
planned_path_component = DrillingCampaign_V1_0_0_Planned_Path_NaturalDeviation.from_dict(
    save_float_array6(natural_deviation_df)
 )

planned_component = DrillingCampaign_V1_0_0_Planned(
    collar=planned_collar_component,
    path=planned_path_component,
 )

display(sorted_planned_df.head())
display(natural_deviation_df.head())

### 6. Build the Interim Section

The interim survey path uses measured depth, azimuth, and dip. Any survey rows whose hole IDs do not appear in the planned collar table are removed so the final drilling campaign stays internally consistent.

In [ ]:
filtered_interim_df = interim_df[interim_df[object_hole_id].isin(hole_id_table_df["value"])].copy()
filtered_interim_df = filtered_interim_df.sort_values([object_hole_id, "depth"]).reset_index(drop=True)

interim_hole_values_df = pd.DataFrame({"data": filtered_interim_df[object_hole_id].astype("string")})
interim_hole_chunks_df = create_hole_id_mapping(hole_id_table_df, interim_hole_values_df)
interim_hole_chunks_component = HoleChunks_V1_0_0.from_dict(data_client.save_dataframe(interim_hole_chunks_df))

interim_survey_df = filtered_interim_df[["depth", "azimuth", "dip"]].rename(columns={"depth": "distance"})
interim_path_component = DownholeDirectionVector_V1_0_0.from_dict(save_float_array3(interim_survey_df))

interim_attributes = build_attribute_components(filtered_interim_df, interim_attribute_specs)
interim_collar_component = HoleCollars_V1_0_0(
    coordinates=coordinates_component,
    distances=distances_component,
    holes=interim_hole_chunks_component,
    attributes=interim_attributes,
)

interim_component = DrillingCampaign_V1_0_0_Interim(
    collar=interim_collar_component,
    path=interim_path_component,
 )

print(f"Retained {len(filtered_interim_df)} interim survey rows after matching hole IDs.")
display(filtered_interim_df.head())

### 7. Create and Publish the Drilling Campaign

Once the planned and interim sections are ready, create the final `DrillingCampaign_V1_0_0` object, upload its referenced Parquet tables, and publish the completed object to Evo.

Because the current API returns object metadata rather than a typed notebook object, the Viewer link is generated with `get_viewer_url_for_objects(...)`.

In [ ]:
drilling_campaign = DrillingCampaign_V1_0_0(
    name=object_name,
    uuid=None,
    bounding_box=bounding_box,
    coordinate_reference_system=coordinate_reference_system,
    hole_id=hole_id_component,
    planned=planned_component,
    interim=interim_component,
    tags=object_tags,
 )

await data_client.upload_referenced_data(drilling_campaign.as_dict(), FeedbackWidget("Uploading data"))
new_drilling_campaign_metadata = await object_client.create_geoscience_object(
    full_obj_path, drilling_campaign.as_dict()
 )

# Generate a viewer
viewer_url = get_viewer_url_for_objects(manager, [new_drilling_campaign_metadata])
print(f"View in Evo Viewer: {viewer_url}")

new_drilling_campaign_metadata

### 8. Summary

This notebook now follows the newer tutorial style used in the typed examples while keeping the current drilling campaign creation API.

- Loaded the planned and interim drilling tables from `sample-data`.
- Built the hole ID lookup, collar data, natural deviation path, and interim survey path from reusable helper functions.
- Created a single drilling campaign object containing both the planned and interim sections.
- Uploaded the referenced Parquet data and published the finished object to Evo.